In [18]:
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score, KFold, RepeatedKFold
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import make_column_transformer, TransformedTargetRegressor
from sklearn.pipeline import make_pipeline,make_union
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, median_absolute_error, mean_squared_error, make_scorer
from sklearn.preprocessing import PowerTransformer, QuantileTransformer, KBinsDiscretizer, StandardScaler
from sklearn.feature_selection import VarianceThreshold, SelectKBest, mutual_info_regression
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, GradientBoostingRegressor, BaggingRegressor

from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.graphics.tsaplots import plot_pacf
from pandas.plotting import autocorrelation_plot, lag_plot

warnings.filterwarnings('ignore')

In [19]:
df=pd.read_csv("training_dataset.csv")
df.drop(["region__peak_sales_dollar_amt_per_hour_v2","region__peak_returns_dollar_amt_per_hour_v2","Unnamed: 0","observation_id","observation_timestamp"], axis=1, inplace=True)
df.head()

,hour_of_day,register__sales_dollar_amt_this_hour,register__payment_types_accepted,register__peak_sales_dollar_amt_per_hour,register__sales_dollar_amt_last_hour,register__sales_quantity_last_hour,register__sales_quantity_rescanned_frac,register__sales_payments_declined_frac,register__peak_returns_dollar_amt_per_hour,register__returns_dollar_amt_last_hour,...,region__mean_service_time_per_customer,region__stdev_service_time_per_customer,region__sales_dollar_amt_last_hour,region__returns_dollar_amt_last_hour,region__nighttime_open_registers,region__nighttime_service_time_per_customer,region__nighttime_sales_amt_per_hour,region__nighttime_returns_amt_per_hour,region__peak_sales_dollar_amt_per_hour,region__peak_returns_dollar_amt_per_hour
0,15,347.29,Cash+Credit,-0.7383,-0.1270,-0.1993,-0.8299,-0.1247,-0.5721,-0.2582,...,-0.9255,-0.3891,-0.6920,-0.4605,-0.5180,-1.0062,-0.6462,-0.6030,-0.4773,-1.7951
1,21,361.59,Cash+Credit,0.6483,-0.0362,-0.0777,-0.7395,-0.1135,-0.5251,-0.2157,...,0.7815,0.0663,-0.6531,-0.4434,-0.6498,0.9031,-0.6493,-0.6106,0.4998,0.8939
2,11,850.73,Cash+Credit,-0.4950,-0.1268,-0.1974,1.3139,0.1075,-0.5251,-0.2579,...,-1.2559,-0.4826,-0.6120,-0.1786,-0.6040,-1.4229,-0.6456,-0.6037,1.9337,2.4046
3,10,1175.69,Cash+Credit,-0.5594,-0.1270,-0.1991,-0.8299,-0.1247,-0.5251,-0.2582,...,-1.2280,-0.4809,-0.5472,0.1246,-0.5925,-1.4097,-0.6478,-0.6079,0.1288,-0.4983
4,14,3204.53,Cash+Credit,0.5693,-0.1221,-0.1632,-0.7071,-0.1247,0.7604,-0.2565,...,0.2442,-0.3021,-0.3838,0.1996,-0.5696,1.1231,-0.6475,-0.6032,1.0050,1.5660


In [23]:
with open("used_columns.txt","r") as file:
    used_columns=[i.replace("\n","") for i in file.readlines()]

# used_columns.extend(df.select_dtypes("object").columns)
# used_columns.append("register__payment_types_accepted")
X=df[used_columns]
y=df.register__sales_dollar_amt_this_hour

In [171]:
with open("note.txt","r") as file:
    skewed_features=[i.replace("\n","") for i in file.readlines()]

In [172]:
for i in train_df.columns:
    print("****************",i,"********************")
    print("Mean:",train_df[i].mean())
    print("Median",train_df[i].median())
    print("Mode:",train_df[i].mode())
    print("Variance:",train_df[i].var())

**************** register__peak_sales_dollar_amt_per_hour ********************
Mean: 0.0029439120570442854
Median -0.3357
Mode: 0   -0.6241
dtype: float64
Variance: 1.0150532957180043
**************** register__sales_quantity_rescanned_frac ********************
Mean: -0.014365622299049637
Median -0.36835
Mode: 0   -0.8299
dtype: float64
Variance: 0.9770889261553838
**************** cashier__title_level ********************
Mean: -0.0018294943820226551
Median -0.352
Mode: 0   -0.352
dtype: float64
Variance: 1.010943162317104
**************** cashier__n_years_experience ********************
Mean: -0.02037705812447196
Median -0.7454
Mode: 0   -0.7454
dtype: float64
Variance: 1.0005881149665965
**************** cashier__hours_into_shift ********************
Mean: 0.0059446737251512595
Median -0.3474
Mode: 0   -0.7183
dtype: float64
Variance: 0.9856359044716575
**************** cashier__item_scan_rate_per_min ********************
Mean: -0.0012299805531550143
Median 0.0913
Mode: 0    1.0186


In [173]:
def detect_outliers_zscore(data):
    outliers=[]
    thres = 3
    mean = np.mean(data)
    std = np.std(data)
    # print(mean, std)
    for i in data:
        z_score = (i-mean)/std
        if (np.abs(z_score) > thres):
            outliers.append(i)
    return outliers

In [174]:
for i in train_df.columns:
    print("****************",i,"********************")
    sample_outliers = detect_outliers_zscore(train_df[i])
    print("Outliers from Z-scores method: ", len(sample_outliers))

**************** register__peak_sales_dollar_amt_per_hour ********************
Outliers from Z-scores method:  411
**************** register__sales_quantity_rescanned_frac ********************
Outliers from Z-scores method:  253
**************** cashier__title_level ********************
Outliers from Z-scores method:  0
**************** cashier__n_years_experience ********************
Outliers from Z-scores method:  0
**************** cashier__hours_into_shift ********************
Outliers from Z-scores method:  658
**************** cashier__item_scan_rate_per_min ********************
Outliers from Z-scores method:  97
**************** cashier__item_manual_entry_rate_per_min ********************
Outliers from Z-scores method:  191
**************** store__miles_to_nearest_location ********************
Outliers from Z-scores method:  0
**************** store__target_sales_quantity_per_hour ********************
Outliers from Z-scores method:  253
**************** store__mean_service_time_

In [11]:
def regression_accuracy(y_meas, y_pred, max_error=20, error_type='relative'):
    '''Compares predicted & measured values, returning the percentage of predictions
       that are within a set error limit. This error limit can be an absolute value
       or a relative percentage'''

    # OPTION 1: Relative percentage
    if ( error_type == 'relative' ):
        mask = 100.0 * abs((y_pred - y_meas) / y_meas) < max_error

    # OPTION 2: Absolute value
    if ( error_type == 'absolute' ):
        mask = abs(y_pred - y_meas) < max_error

    accuracy = sum(mask) / len(mask)

    return accuracy


scorer=make_scorer(regression_accuracy)

In [176]:
class OutliersHandler(BaseEstimator, TransformerMixin):
    def __init__(self, thres=3, method='mean',columns=None):
        self.thres=thres
        self.method=method
        self.columns=columns
    
    def fit(self, X, y=None):
        return self
    def transform(self, X, y=None):
        cols_to_transform = list(X.columns)
        if self.columns:
            cols_to_transform = self.columns
        for col in cols_to_transform:
            X[col]=self.replace_outliers_zscore(X[col],self.thres,self.method)
        return X
    def replace_outliers_zscore(self,data,thres,method):
        new_data=[]
        mean = np.mean(data)
        std = np.std(data)
        if method=='median':
            replace_value=np.median(data)
        if method=='mean':
            replace_value=mean
        for i in data:
            z_score = (i-mean)/std
            if (np.abs(z_score) > thres):
                new_data.append(replace_value)
            else:
                new_data.append(i)
        return new_data

In [12]:
# def make_model(df, estimator=RandomForestRegressor(random_state=0)):
#     numerical_features=df.select_dtypes(["float","int"]).columns

#     numerical_pipeline=make_pipeline(PowerTransformer())
    
#     process=make_column_transformer((numerical_pipeline, numerical_features))

#     model=make_pipeline(process, estimator)

#     return model
def make_model(df, estimator=RandomForestRegressor(random_state=0)):
    numerical_features=df.select_dtypes(["float","int"]).columns
    categorical_features=df.select_dtypes("object").columns

    numerical_pipeline=make_pipeline(StandardScaler())
    categorical_pipeline=make_pipeline(OneHotEncoder(handle_unknown='ignore'))

    process=make_column_transformer((numerical_pipeline, numerical_features), (categorical_pipeline, categorical_features))

    model=make_pipeline(process, estimator)

    return model

In [17]:
%%time
estimator=DecisionTreeRegressor(random_state=0)
# estimator=ExtraTreesRegressor(random_state=21,bootstrap=True)
# estimator=LGBMRegressor(boosting_type='gbdt', objective='regression', num_leaves=1200,
#                                  learning_rate=0.17, n_estimators=30, max_depth=16,
#                                  metric='mae', bagging_fraction=0.8, feature_fraction=0.8, reg_lambda=0.9, use_missing=False)
#model=make_model(X, estimator)

cv=KFold(n_splits=5,shuffle=True,random_state=0)
scores=cross_val_score(estimator, X, y, cv=cv, scoring=scorer, n_jobs=-1)
print(f"CV_SCORES_mean: {scores.mean()}")
print(f"CV_SCORES_std: {scores.std()}")

CV_SCORES_mean: nan
CV_SCORES_std: nan
Wall time: 663 ms


In [186]:
hyper_params = {
    'extratreesregressor__max_depth': range(5,15),
    'extratreesregressor__min_samples_split': range(5,15),
    'extratreesregressor__max_features': range(10, 28),
    'extratreesregressor__max_samples': range(1,10),
    'extratreesregressor__n_estimators': [40,60,100,200,500,1000,1000]
}

In [179]:
model.fit(X,y)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('pipeline',
                                                  Pipeline(steps=[('powertransformer',
                                                                   PowerTransformer())]),
                                                  Index(['register__peak_sales_dollar_amt_per_hour',
       'register__sales_quantity_rescanned_frac', 'cashier__title_level',
       'cashier__n_years_experience', 'cashier__hours_into_shift',
       'cashier__item_scan_rate_per_min',
       'cas...
       'region__nighttime_open_registers',
       'region__nighttime_service_time_per_customer',
       'region__nighttime_returns_amt_per_hour',
       'region__peak_sales_dollar_amt_per_hour',
       'region__peak_returns_dollar_amt_per_hour'],
      dtype='object'))])),
                ('extratreesregressor',
                 ExtraTreesRegressor(max_depth=14, max_features=17,
                                     min_s

In [180]:
test_df=pd.read_csv("test_dataset.csv")
test_df.drop(["region__peak_sales_dollar_amt_per_hour_v2","region__peak_returns_dollar_amt_per_hour_v2"], axis=1, inplace=True)
test_df.head()

,observation_id,observation_timestamp,hour_of_day,register__payment_types_accepted,register__peak_sales_dollar_amt_per_hour,register__sales_dollar_amt_last_hour,register__sales_quantity_last_hour,register__sales_quantity_rescanned_frac,register__sales_payments_declined_frac,register__peak_returns_dollar_amt_per_hour,...,region__mean_service_time_per_customer,region__stdev_service_time_per_customer,region__sales_dollar_amt_last_hour,region__returns_dollar_amt_last_hour,region__nighttime_open_registers,region__nighttime_service_time_per_customer,region__nighttime_sales_amt_per_hour,region__nighttime_returns_amt_per_hour,region__peak_sales_dollar_amt_per_hour,region__peak_returns_dollar_amt_per_hour
0,3f483640-bc52-11ec-b736-8544dc068949,15/04/2022 00:23,0,Cash+Credit,0.5693,-0.1253,-0.1489,-0.8176,-0.1066,0.9103,...,-0.1320,0.3398,-0.6373,-0.3791,-0.5581,-0.7148,-0.6487,-0.6091,1.0020,0.8761
1,a88b0bb0-d2ae-11ec-bd0e-f5a7c7895456,13/05/2022 11:20,11,Cash+Credit,0.3084,-0.1270,-0.1986,1.6562,-0.1247,-0.5251,...,0.8413,0.1972,-0.6483,-0.4419,-0.6498,0.8867,-0.6493,-0.6106,-1.9779,-0.1311
2,90611780-d607-11ec-a5d8-675445c92326,17/05/2022 17:34,17,Cash+Credit,-0.6757,0.1925,0.2534,-0.7517,-0.1247,-0.5251,...,-0.3185,-0.4461,0.6712,0.3971,2.0094,0.0464,1.8275,2.0689,0.0332,-0.3489
3,a98380e0-e2d2-11ec-b28f-7f4d381958ee,03/06/2022 00:18,0,Cash+Credit,-0.1747,-0.1258,-0.1709,-0.3576,-0.1247,-0.5251,...,0.8032,0.1548,-0.6653,-0.4437,-0.6498,0.8937,-0.6493,-0.6106,0.4998,0.8939
4,608b8e60-e4f2-11ec-8115-d3258731976b,05/06/2022 17:10,17,Cash+Credit,0.6483,-0.0612,-0.0995,0.1060,-0.1247,-0.5251,...,0.9011,0.1806,-0.6495,-0.4418,-0.6498,0.9006,-0.6493,-0.6106,0.4998,0.8939


In [181]:
test_df=test_df[X.columns]
predictions = pd.DataFrame(model.predict(test_df), columns=["prediction"])
predictions.head()

,prediction
0,1814.806471
1,352.425233
2,1393.376384
3,232.764663
4,356.863151


In [182]:
predictions.to_csv("submit_35.csv", index=False)